In [ ]:
# import libraries

import sys
import warnings
import os
os.environ['USE_PYGEOS'] = '0'

import geopandas as gpd
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

sys.path.append('../../utils')
import data_paths
from gridding import GriddingEngine

warnings.filterwarnings("ignore")

# Reload local modules on changes
%reload_ext autoreload
%autoreload 2

In [ ]:

filename = 'linesource_Munich_2019_los_specific.gpkg'
gdf = gpd.read_file(data_paths.INVENTORY_PATH+filename)

In [ ]:
species = 'NOx'

gdf['road_length'] = gdf.geometry.length * 1e-3  # convert to km
gdf[f'{species}_total'] = (gdf[f'PC_{species}'] + gdf[f'LCV_{species}'] + \
    gdf[f'HGV_{species}'] + gdf[f'BUS_{species}'] + gdf[f'MOT_{species}'])
gdf[f'{species}_total'] = gdf[f'{species}_total'] * gdf.geometry.length * 1e-9 # convert to t /km

print(f'Total {species}_total:', gdf[f'{species}_total'].sum())
print('\nShare by road type:')
print(gdf.groupby('road_type')[f'{species}_total'].sum() / gdf[f'{species}_total'].sum() * 100)

In [ ]:
path = '/Users/daniel_tum/Documents/code/drive-inventory/data/restricted_input/counting_data/counting_data_combined_until2024.parquet'

df_counts = pd.read_parquet(path)

## Number of individual counters per road type

In [ ]:
df_counts[(df_counts['valid'] == True)].groupby('scaling_road_type')['road_link_id'].nunique()

In [ ]:
df_counts.head()

## Eval number of days with imputation

In [ ]:
df_med = df_counts[(df_counts['complete']) & (df_counts['valid'])].groupby(['vehicle_class',
                                                                   'scaling_road_type',
                                                                   'date'])['daily_value'].median()

df_med = pd.DataFrame(df_med).reset_index()
df_med['year'] = pd.DatetimeIndex(df_med['date']).year
#df_med.set_index(['vehicle_class', 'year', 'scaling_road_type'], inplace=True)

In [ ]:

total_days = len(pd.date_range(start = f'{2020}-01-01', end = f'{2020}-12-31', freq = 'D'))
min_day = (df_med[df_med['year'].isin([2020])].groupby(['vehicle_class', 'scaling_road_type'])['date'].nunique())

share = min_day - total_days


pd.DataFrame(share).reset_index().pivot(index='scaling_road_type', columns='vehicle_class')



In [ ]:
total_days